# Explore "myfiles"

Generated for a selected volume/folder in Catalog Explorer. This notebook shows how to do the following: - viewing file metadata - filtering files by type - filtering files by size and date - processing files with AI functions - setting up Auto Loader

In [0]:
# System-filled
TARGET_PATH = "/Volumes/dbacademy/get_started_de/myfiles"

## 1. List files with metadata

In [0]:
df = spark.read.format("binaryFile").load(TARGET_PATH)
display(df.select("path", "modificationTime", "length"))

## 2. Filter files by type Use `pathGlobFilter` to filter by file type.

In [0]:
df_pdfs = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.{pdf,PDF}")
    .load(TARGET_PATH)
)

display(df_pdfs.select("path", "length"))

## 3. Filter files by size and date Use `length` and `modificationTime` to filter files by size and last modified time.

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr

df_files = (
    spark.read.format("binaryFile")
    .load(TARGET_PATH)
)

filtered_df = (
    df_files
    .filter(
        (col("length").between(20000, 1000000)) &
        (col("modificationTime") >= current_timestamp() - expr("INTERVAL 7 DAY"))
    )
    .orderBy(col("modificationTime").desc())
)

display(filtered_df.select("path", "length", "modificationTime"))

## 4. Process images and documents with AI functions Prerequisite: AI functions require a workspace in a supported region.

In [0]:
# Parse documents (example: PDFs) in your volume/folder
# Escape single quotes so a path like .../o'brien stays a valid SQL string literal.
safe_path = TARGET_PATH.replace("'", "''")
result_df = spark.sql(f"""
    SELECT
      path AS file_path,
      CAST(ai_parse_document(content, map('version', '2.0')) AS STRING) AS parsed_content
    FROM read_files(
      '{safe_path}',
      format => 'binaryFile',
      fileNamePattern => '*.pdf'
    )
""")
display(result_df)

In [0]:
# Analyze images (example: JPG/PNG) in your volume/folder
# Escape single quotes so a path like .../o'brien stays a valid SQL string literal.
safe_path = TARGET_PATH.replace("'", "''")
result_df = spark.sql(f"""
    SELECT
      path,
      ai_query(
        'databricks-llama-4-maverick',
        'Describe this image in one sentence:',
        files => content
      ) AS description
    FROM read_files(
      '{safe_path}',
      format => 'binaryFile',
      fileNamePattern => '*.{{jpg,jpeg,png}}'
    )
    WHERE _metadata.file_size < 5000000
""")
display(result_df)

## 5. Set up Auto Loader Prerequisite: Review and edit before running, especially the checkpoint path and table name.

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Edit if needed.
INCOMING_PATH = TARGET_PATH
CHECKPOINT_PATH = TARGET_PATH + "_checkpoints/docs/"
OUTPUT_TABLE = "document_ingestion"

df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .load(INCOMING_PATH)
)

df_enriched = (
    df_stream
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

query = (
    df_enriched.writeStream
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(OUTPUT_TABLE)
)

query.awaitTermination()